# Fill Replication Gaps: Salience, Policy Overlap, Bill Sponsorship

This notebook documents the three gap-filling steps that bring the
replication from 88% to full coverage of all three legacy GLMM models.

**Gaps filled:**
1. Policy salience (Google Trends data for 18 CAP policy areas)
2. Policy overlap (member committee assignments vs. org policy area)
3. Bill sponsorship (Congress.gov API member-level counts)

**Scripts used:**
- `scripts/collect_policy_salience.py`
- `scripts/derive_policy_overlap.py`
- `scripts/derive_bill_sponsorship.py`
- `scripts/merge_salience_into_dataset.py`

## 1. Load Updated Analysis Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/output/analysis_dataset_replication.csv', low_memory=False)
print(f'Total rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'\nNew columns added:')
for col in ['salience_score', 'salience_category', 'policy_overlap', 'bills_sponsored']:
    if col in df.columns:
        nn = df[col].notna().sum()
        print(f'  {col:25s}: {nn:,} non-null ({nn/len(df)*100:.1f}%)')
    else:
        print(f'  {col:25s}: NOT PRESENT')

Total rows: 57,073
Columns: 39

New columns added:
  salience_score           : 14,390 non-null (25.2%)
  salience_category        : 14,390 non-null (25.2%)
  policy_overlap           : 12,056 non-null (21.1%)
  bills_sponsored          : 22,248 non-null (39.0%)


## 2. Policy Salience (Google Trends)

In [2]:
sal = pd.read_csv('../../data/input/policy_salience_scores.csv')
print('Policy Salience Scores (Google Trends 2015-2019):')
print(sal[['policy_area', 'salience_score', 'salience_category']]
      .sort_values('salience_score', ascending=False)
      .to_string(index=False))
print(f'\nCategories: {sal["salience_category"].value_counts().to_dict()}')

Policy Salience Scores (Google Trends 2015-2019):
          policy_area  salience_score salience_category
       Social Welfare       54.925397              high
       Transportation       43.752381              high
              Housing       43.695238              high
        Law and Crime       41.926190              high
           Technology       39.669048              high
          Agriculture       37.712698              high
               Health       36.606349            medium
       Macroeconomics       33.607937            medium
            Education       32.833333            medium
         Civil Rights       31.301587            medium
          Environment       30.080952            medium
Government Operations       26.985714            medium
    Domestic Commerce       25.095238               low
              Defense       23.879365               low
International Affairs       20.433333               low
         Public Lands       17.152381               lo

## 3. Policy Overlap (Committee-Policy Match)

In [3]:
if 'policy_overlap' in df.columns:
    mentions = df[df['is_zero_mention'] == 0]
    overlap_data = mentions['policy_overlap'].dropna()
    print(f'Policy overlap available for: {len(overlap_data):,} mentions')
    print(f'Overlap rate: {overlap_data.mean():.3f} ({overlap_data.mean()*100:.1f}%)')
    print(f'\nValue counts:\n{mentions["policy_overlap"].value_counts(dropna=False)}')
    print(f'\nSource: pipeline intermediate granule_committees + granule_members data')
    print(f'Committee-to-policy mapping from committee_policy_linkage.py')
else:
    print('policy_overlap column not present')

Policy overlap available for: 12,056 mentions
Overlap rate: 0.980 (98.0%)

Value counts:
policy_overlap
NaN    41836
1.0    11812
0.0      244
Name: count, dtype: int64

Source: pipeline intermediate granule_committees + granule_members data
Committee-to-policy mapping from committee_policy_linkage.py


## 4. Bill Sponsorship

In [4]:
if 'bills_sponsored' in df.columns and df['bills_sponsored'].notna().sum() > 0:
    mentions = df[df['is_zero_mention'] == 0]
    bs = mentions['bills_sponsored'].dropna()
    print(f'bills_sponsored available for: {len(bs):,} mentions')
    print(f'Stats: {bs.describe().to_dict()}')
    print(f'Source: Congress.gov API (total bills sponsored per member)')
else:
    print('bills_sponsored not available from API.')
    if 'bills_referenced' in df.columns:
        mentions = df[df['is_zero_mention'] == 0]
        br = mentions['bills_referenced'].dropna()
        print(f'\nUsing bills_referenced as fallback:')
        print(f'  Available for: {len(br):,} mentions')
        print(f'  Non-zero: {(br > 0).sum():,} ({(br > 0).mean()*100:.1f}%)')
        print(f'  Stats: {br.describe().to_dict()}')
        print(f'\n  Note: bills_referenced measures speech-level bill citations,')
        print(f'  while legacy bills_sponsored measured member-level total sponsorship.')
        print(f'  These are different constructs; comparison should be cautious.')

bills_sponsored available for: 22,248 mentions
Stats: {'count': 22248.0, 'mean': 805.9819309600863, 'std': 1071.5707875151877, 'min': 11.0, '25%': 149.0, '50%': 303.0, '75%': 1194.0, 'max': 5992.0}
Source: Congress.gov API (total bills sponsored per member)


## 5. GLMM Model Readiness Check

In [5]:
mentions = df[df['is_zero_mention'] == 0].copy()
print('MODEL SAMPLE SIZES')
print('=' * 60)

# Model A: Policy Salience
ma_cols = ['salience_category', 'is_democrat', 'is_senate', 'is_membership_org',
           'is_single_issue', 'is_labor']
ma_available = [c for c in ma_cols if c in mentions.columns]
ma_data = mentions.dropna(subset=ma_available)
print(f'Model A (Policy Salience): {len(ma_data):,} rows')
print(f'  Variables: {ma_available}')

# Model B: Group-Politician Linkage
mb_cols = ['terms_served_before', 'up_for_reelection', 'is_democrat',
           'is_senate', 'log_lobbying']
mb_extra = []
if 'policy_overlap' in mentions.columns:
    mb_cols.append('policy_overlap')
    mb_extra.append('policy_overlap')
if 'bills_sponsored' in mentions.columns and mentions['bills_sponsored'].notna().sum() > 0:
    mb_cols.append('bills_sponsored')
    mb_extra.append('bills_sponsored')
else:
    mb_cols.append('bills_referenced')
    mb_extra.append('bills_referenced (fallback)')
mb_data = mentions.dropna(subset=mb_cols)
print(f'Model B (Group-Politician): {len(mb_data):,} rows')
print(f'  Variables: {mb_cols}')
print(f'  New variables: {mb_extra}')

# Model C: Group Characteristics
mc_cols = ['org_age', 'log_lobbying', 'policy_scope', 'is_single_issue',
           'is_labor', 'is_membership_org']
mc_data = mentions.dropna(subset=mc_cols)
print(f'Model C (Group Characteristics): {len(mc_data):,} rows')
print(f'  Variables: {mc_cols}')

MODEL SAMPLE SIZES
Model A (Policy Salience): 14,390 rows
  Variables: ['salience_category', 'is_democrat', 'is_senate', 'is_membership_org', 'is_single_issue', 'is_labor']
Model B (Group-Politician): 12,037 rows
  Variables: ['terms_served_before', 'up_for_reelection', 'is_democrat', 'is_senate', 'log_lobbying', 'policy_overlap', 'bills_sponsored']
  New variables: ['policy_overlap', 'bills_sponsored']
Model C (Group Characteristics): 42,747 rows
  Variables: ['org_age', 'log_lobbying', 'policy_scope', 'is_single_issue', 'is_labor', 'is_membership_org']


## 6. Final Replication Scorecard

In [6]:
print('=' * 60)
print('FINAL REPLICATION SCORECARD')
print('=' * 60)
print()
print(f'{"Status":15s} {"Before":>10s} {"After":>10s}')
print(f'{"READY":15s} {"15":>10s} {"17":>10s}')
print(f'{"PARTIAL":15s} {"2":>10s} {"0":>10s}')
print(f'{"NOT USABLE":15s} {"0":>10s} {"0":>10s}')
print(f'{"Overall":15s} {"88%":>10s} {"100%":>10s}')
print()
print('VARIABLE STATUS:')
variables = [
    ('prominence (DV)', 'AVAILABLE', 'level1.csv classification'),
    ('chamber', 'AVAILABLE', 'member_profiles'),
    ('party', 'AVAILABLE', 'member_profiles'),
    ('seniority', 'AVAILABLE', 'congress-legislators'),
    ('election_timing', 'AVAILABLE', 'congress-legislators'),
    ('org_type', 'AVAILABLE', 'WRS metadata'),
    ('org_age', 'AVAILABLE', 'WRS FOUNDED field'),
    ('log_lobbying', 'AVAILABLE', 'WRS LOBBYING11'),
    ('policy_scope', 'AVAILABLE', 'derived from level1'),
    ('policy_salience', 'NEW', 'Google Trends via pytrends'),
    ('policy_overlap', 'NEW', 'committee-policy matching'),
    ('bill_engagement', 'NEW/FALLBACK', 'Congress.gov or bills_referenced'),
    ('membership_status', 'AVAILABLE', 'WRS MSHIP_STATUS11'),
]
print(f'{"Variable":25s} {"Status":15s} {"Source"}')
print('-' * 70)
for var, status, source in variables:
    print(f'{var:25s} {status:15s} {source}')
print()
print('MODELS READY TO RUN:')
print('  Model A (Policy Salience):       YES')
print('  Model B (Group-Politician):      YES')
print('  Model C (Group Characteristics): YES')
print()
print('Run in RStudio: Rscript scripts/run_glmm_replication.R')

FINAL REPLICATION SCORECARD

Status              Before      After
READY                   15         17
PARTIAL                  2          0
NOT USABLE               0          0
Overall                88%       100%

VARIABLE STATUS:
Variable                  Status          Source
----------------------------------------------------------------------
prominence (DV)           AVAILABLE       level1.csv classification
chamber                   AVAILABLE       member_profiles
party                     AVAILABLE       member_profiles
seniority                 AVAILABLE       congress-legislators
election_timing           AVAILABLE       congress-legislators
org_type                  AVAILABLE       WRS metadata
org_age                   AVAILABLE       WRS FOUNDED field
log_lobbying              AVAILABLE       WRS LOBBYING11
policy_scope              AVAILABLE       derived from level1
policy_salience           NEW             Google Trends via pytrends
policy_overlap            NEW 